# Analisis Mitra yang Memiliki Dokumen Kerja Sama

Notebook ini menganalisis mitra (distinct) yang bekerja sama dan memiliki dokumen kerja sama.

**Informasi yang ditampilkan:**
- Nama Mitra
- Nomor Telepon Mitra
- Email Mitra
- Nama Penandatangan Mitra
- PIC Mitra (Person In Charge)
- Nomor Telepon PIC
- Email PIC
- Tanggal Mulai Kerja Sama

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import os

# Path ke folder data
data_dir = 'data'

# Load semua tabel yang dibutuhkan
df_mitra = pd.read_csv(os.path.join(data_dir, 'T_mitra - T_mitra.csv'))
df_person = pd.read_csv(os.path.join(data_dir, 'T_person - T_person.csv'))
df_dokumen = pd.read_csv(os.path.join(data_dir, 'T_dokumen_kerjasama - T_dokumen_kerjasama.csv'))
df_pengajuan = pd.read_csv(os.path.join(data_dir, 'T_pengajuan_kerjasama - T_pengajuan_kerjasama.csv'))
df_mapping_mitra = pd.read_csv(os.path.join(data_dir, 'M_mitra_bekerjasama - M_dudi_bekerjasama.csv'))

print(f'Jumlah Mitra: {len(df_mitra)}')
print(f'Jumlah Person: {len(df_person)}')
print(f'Jumlah Dokumen Kerjasama: {len(df_dokumen)}')
print(f'Jumlah Pengajuan Kerjasama: {len(df_pengajuan)}')
print(f'Jumlah Mapping Mitra Bekerjasama: {len(df_mapping_mitra)}')

## 2. Preview Data

In [ ]:
print('=== T_mitra (5 baris pertama) ===')
display(df_mitra.head())

print('\n=== M_mitra_bekerjasama (5 baris pertama) ===')
display(df_mapping_mitra.head())

print('\n=== T_dokumen_kerjasama (5 baris pertama) ===')
display(df_dokumen[['id', 'ref_pengajuan_kerjasama', 'tanggal_mulai', 'tanggal_berakhir', 'nomor_dokumen']].head())

print('\n=== T_person (5 baris pertama) ===')
display(df_person.head())

## 3. Menggabungkan Data

Alur relasi:
1. **M_mitra_bekerjasama** → `ref_mitra` → **T_mitra** (info mitra)
2. **M_mitra_bekerjasama** → `ref_penandatangan_mitra` → **T_person** (penandatangan)
3. **M_mitra_bekerjasama** → `ref_pic_mitra` → **T_person** (PIC mitra)
4. **M_mitra_bekerjasama** → `ref_pengajuan_kerjasama` → **T_dokumen_kerjasama** (tanggal mulai)

In [ ]:
# Step 1: Join mapping mitra dengan data mitra
df_merged = df_mapping_mitra.merge(
    df_mitra[['id', 'mitra', 'email', 'telepon']].rename(columns={
        'id': 'id_mitra',
        'mitra': 'nama_mitra',
        'email': 'email_mitra',
        'telepon': 'telepon_mitra'
    }),
    left_on='ref_mitra',
    right_on='id_mitra',
    how='left'
)
df_merged.drop(columns=['id_mitra'], inplace=True, errors='ignore')

print(f'Setelah join mitra: {len(df_merged)} baris')
df_merged[['nama_mitra', 'email_mitra', 'telepon_mitra']].head()

In [ ]:
# Step 2: Join dengan T_person untuk Penandatangan Mitra
df_merged = df_merged.merge(
    df_person[['id', 'nama', 'email', 'telepon']].rename(columns={
        'id': 'id_penandatangan',
        'nama': 'nama_penandatangan',
        'email': 'email_penandatangan',
        'telepon': 'telepon_penandatangan'
    }),
    left_on='ref_penandatangan_mitra',
    right_on='id_penandatangan',
    how='left'
)
df_merged.drop(columns=['id_penandatangan'], inplace=True, errors='ignore')

print(f'Setelah join penandatangan: {len(df_merged)} baris')
df_merged[['nama_mitra', 'nama_penandatangan']].head()

In [ ]:
# Step 3: Join dengan T_person untuk PIC Mitra
df_merged = df_merged.merge(
    df_person[['id', 'nama', 'email', 'telepon']].rename(columns={
        'id': 'id_pic',
        'nama': 'nama_pic_mitra',
        'email': 'email_pic_mitra',
        'telepon': 'telepon_pic_mitra'
    }),
    left_on='ref_pic_mitra',
    right_on='id_pic',
    how='left'
)
df_merged.drop(columns=['id_pic'], inplace=True, errors='ignore')

print(f'Setelah join PIC: {len(df_merged)} baris')
df_merged[['nama_mitra', 'nama_pic_mitra', 'telepon_pic_mitra', 'email_pic_mitra']].head()

In [ ]:
# Step 4: Join dengan T_dokumen_kerjasama untuk mendapatkan tanggal_mulai
# Relasi: M_mitra_bekerjasama.ref_pengajuan_kerjasama == T_dokumen_kerjasama.ref_pengajuan_kerjasama

# Parse tanggal_mulai
df_dokumen['tanggal_mulai_parsed'] = pd.to_datetime(df_dokumen['tanggal_mulai'], format='mixed', errors='coerce')

# Ambil tanggal_mulai paling awal dan nomor_dokumen per pengajuan
df_dok_agg = df_dokumen.groupby('ref_pengajuan_kerjasama').agg(
    tanggal_mulai=('tanggal_mulai_parsed', 'min'),
    nomor_dokumen=('nomor_dokumen', 'first')
).reset_index()

df_merged = df_merged.merge(
    df_dok_agg,
    left_on='ref_pengajuan_kerjasama',
    right_on='ref_pengajuan_kerjasama',
    how='inner'  # INNER join: hanya mitra yang memiliki dokumen kerjasama
)

print(f'Setelah join dokumen (hanya mitra dengan dokumen): {len(df_merged)} baris')
df_merged[['nama_mitra', 'tanggal_mulai', 'nomor_dokumen']].head()

## 4. Distinct Mitra & Hasil Akhir

Menampilkan mitra yang **distinct** (tidak duplikat) beserta informasi yang diminta.

In [ ]:
# Pilih kolom yang dibutuhkan
kolom_hasil = [
    'nama_mitra',
    'telepon_mitra',
    'email_mitra',
    'nama_penandatangan',
    'nama_pic_mitra',
    'telepon_pic_mitra',
    'email_pic_mitra',
    'tanggal_mulai',
    'nomor_dokumen'
]

df_hasil = df_merged[kolom_hasil].copy()

# Distinct berdasarkan nama_mitra (drop duplicates)
df_distinct = df_hasil.drop_duplicates(subset=['nama_mitra'], keep='first')
df_distinct = df_distinct.sort_values('nama_mitra').reset_index(drop=True)
df_distinct.index = df_distinct.index + 1  # Mulai index dari 1
df_distinct.index.name = 'No'

# Format tanggal
df_distinct['tanggal_mulai'] = df_distinct['tanggal_mulai'].dt.strftime('%Y-%m-%d')

print(f'📊 Total Mitra Distinct yang Memiliki Dokumen Kerja Sama: {len(df_distinct)}')
print('=' * 80)
df_distinct

## 5. Ringkasan Statistik

In [ ]:
print('📈 RINGKASAN STATISTIK')
print('=' * 50)
print(f'Total mitra unik dengan dokumen kerjasama : {len(df_distinct)}')
print(f'Mitra dengan email terisi                 : {df_distinct["email_mitra"].notna().sum()}')
print(f'Mitra dengan telepon terisi                : {df_distinct["telepon_mitra"].notna().sum()}')
print(f'Mitra dengan penandatangan terisi          : {df_distinct["nama_penandatangan"].notna().sum()}')
print(f'Mitra dengan PIC terisi                    : {df_distinct["nama_pic_mitra"].notna().sum()}')
print(f'PIC dengan telepon terisi                  : {df_distinct["telepon_pic_mitra"].notna().sum()}')
print(f'PIC dengan email terisi                    : {df_distinct["email_pic_mitra"].notna().sum()}')

## 6. Export ke Excel (Opsional)

In [ ]:
# Uncomment baris di bawah untuk export ke Excel
# output_file = 'hasil_analisis_mitra_distinct.xlsx'
# df_distinct.to_excel(output_file, index=True)
# print(f'✅ Data berhasil diekspor ke: {output_file}')